## Understanding and Preserving Safety in Fine-Tuned LLMs

This demo presents safety-preserving fine-tuning (SPF), a lightweight approach that explicitly removes gradient components conflicting with the low-rank safety subspace.

<br/>

### Environment Setup

We recommend using Python 3.10. Install dependencies via one of the following two ways:

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
!conda env create -f ../environment.yml
!conda activate spf

<br/>

### Step 1: Normal Fine-tuning of an LLM on a Benign Task

We fine-tune `Qwen/Qwen2.5-7B-Instruct` on the [sql-create-context](https://huggingface.co/datasets/b-mc2/sql-create-context) dataset.

In [11]:
!CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 cd .. && accelerate launch --config_file=accelerate_configs/deepspeed_zero2.yaml \
  --num_processes 8 \
  finetune.py --model_name_or_path='Qwen/Qwen2.5-7B-Instruct' \
  --dataset_name='sql_create_context' --model_family='qwen2' --learning_rate=2e-5 \
  --per_device_train_batch_size=16 --gradient_accumulation_steps=1 \
  --output_dir='ft_paper/outputs/sql/qwen_25_7b' \
  --logging_steps=1 --num_train_epochs=3 --gradient_checkpointing --report_to=none \
  --torch_dtype=bfloat16 --bf16=True --bf16_full_eval=True --save_strategy='no' \
  --sft_type='sft' \
  --use_warmup=True ;

[2026-01-13 16:33:00,761] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:33:01,360] torch.distributed.run: [WARNING] 
[2026-01-13 16:33:01,360] torch.distributed.run: [WARNING] *****************************************
[2026-01-13 16:33:01,360] torch.distributed.run: [WARNING] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[2026-01-13 16:33:01,360] torch.distributed.run: [WARNING] *****************************************
[2026-01-13 16:33:04,063] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:33:04,110] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:33:04,110] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (au

### Step 2: Evaluate the Harmfulness and Task Utility of the Model after Normal Fine-tuning

After fine-tuning on a benign task, the model's safety alignment is compromised, although its task performance becomes high. We use the [HarmBench](https://arxiv.org/abs/2402.04249) classifier to evaluate the attack success rate (ASR), and Rouge-1 scoring on the benign dataset's test set to evaluate task utility.

In [8]:
import json

<br/>

#### Baseline ASR/Utility

In [12]:
!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch  --num_processes=1 \
	eval_safety.py \
	--torch_dtype=bfloat16 --model_name_or_path="Qwen/Qwen2.5-7B-Instruct" \
	--safety_bench='hex-phi' --model_family='qwen2' \
  	--prompt_style='qwen2' --evaluator='harmbench' \
  	--save_path='results/asr/qwen_25_7b.json' --eval_template='pure_bad';

!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch --num_processes=1 \
	eval_utility.py \
	--torch_dtype=bfloat16 --model_name_or_path='Qwen/Qwen2.5-7B-Instruct' \
	--dataset='sql_create_context' --model_family='qwen2' \
	--prompt_style='qwen2' --evaluator='rouge_1' \
	--save_path="results/util/qwen_25_7b.json";

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 16:42:22,143] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Evaluating with HarmBench: 100%|████████████████| 21/21 [00:41<00:00,  1.98s/it]
{'evaluator': 'harmbench', 'num_tot': 330, 'num_success': 40, 'asr': 0.12121212121212122}
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 16:50:00,072] [INFO] 

Harmfulness:

In [13]:
with open("../results/asr/qwen_25_7b.json", "r") as f:
    data = json.load(f)
    print(data["metrics"]["asr"])

0.12121212121212122


Task Performance:

In [14]:
with open("../results/util/qwen_25_7b.json", "r") as f:
    data = json.load(f)
    print(data["metrics"][2])

0.7624605522377653


<br/>

#### ASR/Utility after Normal Fine-tuning

In [15]:
!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch --num_processes=1 \
	eval_safety.py \
	--torch_dtype=bfloat16 --model_name_or_path="outputs/sql/qwen_25_7b" \
	--safety_bench='hex-phi' --model_family='qwen2' \
  	--prompt_style='qwen2' --evaluator='harmbench' \
  	--save_path='results/asr/qwen_25_7b_sql.json' --eval_template='pure_bad';

!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch --num_processes=1 \
	eval_utility.py \
	--torch_dtype=bfloat16 --model_name_or_path='outputs/sql/qwen_25_7b' \
	--dataset='sql_create_context' --model_family='qwen2' \
	--prompt_style='qwen2' --evaluator='rouge_1' \
	--save_path="results/util/qwen_25_7b_sql.json";

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 16:57:58,575] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Evaluating with HarmBench: 100%|████████████████| 21/21 [00:36<00:00,  1.73s/it]
{'evaluator': 'harmbench', 'num_tot': 330, 'num_success': 72, 'asr': 0.21818181818181817}
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 17:03:57,881] [INFO] 

In [ ]:
CUDA_VISIBLE_DEVICES=5,6 cd .. && accelerate launch --config_file=accelerate_configs/deepspeed_zero2.yaml \
  --num_processes 2 \
  --main_process_port 29501 \
  finetune.py --model_name_or_path='Qwen/Qwen2.5-7B-Instruct' \
  --dataset_name='pure_bad' --model_family='qwen2' --learning_rate=2e-5 \
  --per_device_train_batch_size=16 --gradient_accumulation_steps=1 \
  --output_dir='outputs/pure_bad/qwen_25_7b_spf' \
  --logging_steps=1 --num_train_epochs=5 --gradient_checkpointing --report_to=none \
  --torch_dtype=bfloat16 --bf16=True --bf16_full_eval=True --save_strategy='no' \
  --sft_type='sft' \
  --use_warmup=True \
  --use_spf=True


[2026-03-05 18:05:45,411] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-05 18:05:45,886] torch.distributed.run: [WARNING] 
[2026-03-05 18:05:45,886] torch.distributed.run: [WARNING] *****************************************
[2026-03-05 18:05:45,886] torch.distributed.run: [WARNING] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[2026-03-05 18:05:45,886] torch.distributed.run: [WARNING] *****************************************
[2026-03-05 18:05:48,636] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-03-05 18:05:48,668] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
/c23030/ckj/miniconda3/envs/reasoning/lib/python3.10/site-packages/transformers/utils/import_utils.py:560: Fu

Harmfulness:

In [16]:
with open("../results/asr/qwen_25_7b_sql.json", "r") as f:
    data = json.load(f)
    print(data["metrics"]["asr"])

0.21818181818181817


Task Performance:

In [17]:
with open("../results/util/qwen_25_7b_sql.json", "r") as f:
    data = json.load(f)
    print(data["metrics"][2])

0.9928248874140447


<br/>

### Step 3: Safety-preserving Fine-tuning

During fine-tuning, we calculate the gradient of a representative safety example. We then orthogonalize the utility updates with respect to the safety direction by subtracting the projection of the utility gradients onto the safety gradient. This mitigates the impact of utility data on safety parameters, preserving alignment without compromising task performance.

In [5]:
!CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 cd .. && accelerate launch --config_file=accelerate_configs/deepspeed_zero2.yaml \
  --num_processes 8 \
  finetune.py --model_name_or_path='Qwen/Qwen2.5-7B-Instruct' \
  --dataset_name='sql_create_context' --model_family='qwen2' --learning_rate=2e-5 \
  --per_device_train_batch_size=16 --gradient_accumulation_steps=1 \
  --output_dir='outputs/sql/qwen_25_7b_spf' \
  --logging_steps=1 --num_train_epochs=3 --gradient_checkpointing --report_to=none \
  --torch_dtype=bfloat16 --bf16=True --bf16_full_eval=True --save_strategy='no' \
  --sft_type='sft' \
  --use_warmup=True \
  --use_spf=True;

[2026-01-13 16:12:43,249] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:12:43,879] torch.distributed.run: [WARNING] 
[2026-01-13 16:12:43,879] torch.distributed.run: [WARNING] *****************************************
[2026-01-13 16:12:43,879] torch.distributed.run: [WARNING] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
[2026-01-13 16:12:43,879] torch.distributed.run: [WARNING] *****************************************
[2026-01-13 16:12:46,583] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:12:46,639] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2026-01-13 16:12:46,642] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (au

<br/>

### Step 4: Evaluate the Harmfulness and Task Utility of the Model After Safety-preserving Fine-tuning

In [6]:
!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch --num_processes=1 \
	eval_safety.py \
	--torch_dtype=bfloat16 --model_name_or_path="outputs/sql/qwen_25_7b_spf" \
	--safety_bench='hex-phi' --model_family='qwen2' \
  	--prompt_style='qwen2' --evaluator='harmbench' \
  	--save_path='results/asr/qwen_25_7b_spf.json' --eval_template='pure_bad';

!cd .. && CUDA_VISIBLE_DEVICES=7 accelerate launch --num_processes=1 \
	eval_utility.py \
	--torch_dtype=bfloat16 --model_name_or_path='outputs/sql/qwen_25_7b_spf' \
	--dataset='sql_create_context' --model_family='qwen2' \
	--prompt_style='qwen2' --evaluator='rouge_1' \
	--save_path="results/util/qwen_25_7b_spf.json";

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 16:26:33,477] [INFO] [real_accelerator.py:191:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Evaluating with HarmBench: 100%|████████████████| 21/21 [00:14<00:00,  1.45it/s]
{'evaluator': 'harmbench', 'num_tot': 330, 'num_success': 5, 'asr': 0.015151515151515152}
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[2026-01-13 16:28:12,798] [INFO] 

Harmfulness:

In [9]:
with open("../results/asr/qwen_25_7b_spf.json", "r") as f:
    data = json.load(f)
    print(data["metrics"]["asr"])

0.015151515151515152


Task Performance:

In [10]:
with open("../results/util/qwen_25_7b_spf.json", "r") as f:
    data = json.load(f)
    print(data["metrics"][2])

0.9922614908002775
